# MLOps Lab 7: Serving a Model as a Web Service with FastAPI

## 1. Aim

To make a trained machine learning model available to other software by building a web service with FastAPI that validates incoming data and returns predictions.

## 2. Theory

### 2.1 The problem

A model that exists only inside a notebook cannot be used by anyone else. A food-delivery app, for example, needs to send the details of an order and receive a predicted delivery time. To make this possible, the model is placed behind a **web service**.

### 2.2 APIs and REST

Software communicates with other software through an **API** (Application Programming Interface). A web API has four basic parts:

- **Endpoint**: an address on the service, for example `/predict`
- **Method**: `GET` fetches information; `POST` sends data and receives a result
- **Body**: the data sent with a request, in **JSON** format, which looks like a Python dictionary
- **Status code**: the outcome of the request. `200` means success, `422` means the input was invalid, and `500` means the service failed

### 2.3 FastAPI

FastAPI is a Python framework for building web services. It **validates input automatically**, rejecting requests with missing or impossible values before they reach the model, and it **generates a documentation page** at `/docs`, where every endpoint can be tried in the browser.

Input validation matters because a model does not know when its input is unreasonable. A model trained on distances of 1 to 15 km will happily return a meaningless prediction for 500 km.

## 3. Procedure Overview

1. Train a model and save it
2. Write the web service
3. Test the service inside the notebook
4. Observe how invalid input is rejected
5. Run the service as a real server and open the documentation page
6. Complete the task: Iris Flower Classification Service

**Installation note.** If FastAPI is not yet installed, run the following command once in a terminal:

```
pip install fastapi uvicorn httpx
```

## Step 1: Train a Model and Save It

We create a synthetic food-delivery dataset of 500 orders with three features: `distance_km` (1 to 15 km), `items` (1 to 9) and `traffic_level` (1 = low, 2 = medium, 3 = high). We train a Linear Regression model and save it to `work/model.joblib`. The web service will load the model from this file.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression

# Create the dataset (fixed seed so everyone gets the same data)
np.random.seed(42)
n = 500

df = pd.DataFrame()
df["distance_km"]   = np.random.uniform(1, 15, n)
df["items"]         = np.random.randint(1, 10, n)
df["traffic_level"] = np.random.randint(1, 4, n)
df["delivery_minutes"] = (10 + 3 * df["distance_km"] + 1.5 * df["items"]
                          + 5 * df["traffic_level"] + np.random.normal(0, 3, n))

# Train the model
X = df[["distance_km", "items", "traffic_level"]]
y = df["delivery_minutes"]

model = LinearRegression()
model.fit(X, y)

# Save the trained model in the "work" folder
os.makedirs("work", exist_ok=True)
joblib.dump(model, "work/model.joblib")
print("Model saved to work/model.joblib")

Model saved to work/model.joblib


## Step 2: Write the Web Service

The service is saved as a Python file, `work/app.py`, using the command `%%writefile`. It contains:

- **`Order`**: a Pydantic class that describes a valid request. `Field(ge=..., le=...)` sets the allowed range (`ge` = greater than or equal to, `le` = less than or equal to, `gt` = greater than).
- **`GET /health`**: returns `{"status": "ok"}` to show that the service is running.
- **`POST /predict`**: receives an order and returns the predicted delivery time.

In [2]:
%%writefile work/app.py
import os
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Load the saved model once, when the service starts
model = joblib.load(os.path.join(os.path.dirname(__file__), "model.joblib"))

app = FastAPI(title="Delivery Time Prediction Service")


# The shape of a valid request (ranges match the training data)
class Order(BaseModel):
    distance_km:   float = Field(gt=0, le=15)
    items:         int   = Field(ge=1, le=9)
    traffic_level: int   = Field(ge=1, le=3)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(order: Order):
    # Convert the order into a one-row table, as the model expects
    row = pd.DataFrame([order.model_dump()])
    prediction = model.predict(row)[0]
    return {"predicted_minutes": round(float(prediction), 1)}

Writing work/app.py


## Step 3: Test the Service Inside the Notebook

FastAPI's **TestClient** sends requests directly to the application, so no server is needed. We call both endpoints and print the status code and reply.

In [3]:
import sys
import importlib
from fastapi.testclient import TestClient

# Load app.py from the "work" folder (reload picks up any edits to the file)
sys.path.insert(0, "work")
import app
importlib.reload(app)

client = TestClient(app.app)

# Check that the service is running
response = client.get("/health")
print("Health  :", response.status_code, response.json())

# Ask for a prediction
order = {"distance_km": 8, "items": 3, "traffic_level": 2}
response = client.post("/predict", json=order)
print("Predict :", response.status_code, response.json())

Health  : 200 {'status': 'ok'}
Predict : 200 {'predicted_minutes': 48.4}


## Step 4: Observe How Invalid Input Is Rejected

We send an order with an impossible traffic level (9). The service replies with status code **422** and explains the problem. The model is never called.

In [4]:
bad_order = {"distance_km": 8, "items": 3, "traffic_level": 9}

response = client.post("/predict", json=bad_order)

print("Status code:", response.status_code)
print("Problem    :", response.json()["detail"][0]["msg"])

Status code: 422
Problem    : Input should be less than or equal to 3


## Step 5: Run the Service as a Real Server

We now start the service as a real web server using **uvicorn**. **Run this cell only once**; running it again fails because port 8000 is already in use.

After about 5 seconds, open **http://127.0.0.1:8000/docs** in a web browser. Open **POST /predict**, click **Try it out**, enter an order, and click **Execute**. The prediction appears in the **Response body** panel.

**Alternative.** The server can also be started from a terminal opened in the `work` folder:

```
python -m uvicorn app:app --port 8000
```

> **Windows users:** use **Anaconda Prompt**, move to the `work` folder using `cd`, and run the command above. If a firewall prompt appears, click **Allow**.

In [5]:
import subprocess

# Start the web server in the background, inside the "work" folder
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--port", "8000"],
    cwd="work",
)

print("Server is starting. Open http://127.0.0.1:8000/docs in your browser.")

Server is starting. Open http://127.0.0.1:8000/docs in your browser.


To stop the server at the end of the session, remove the `#` and run the cell below.

In [6]:
# server.terminate()

# Task: Iris Flower Classification Service

Use the **Iris dataset**, which is built into scikit-learn (load it with `from sklearn.datasets import load_iris`, so no download is needed), to build a web service that predicts the species of an iris flower from four measurements in centimetres: sepal length, sepal width, petal length and petal width. Train a `LogisticRegression(max_iter=1000)` model, save it in a folder named `task`, and write a FastAPI service in `task/iris_app.py` with two endpoints: `GET /health`, which returns `{"status": "ok"}`, and `POST /predict`, which accepts the four measurements and returns the predicted species **name** (for example `"setosa"`). Use Pydantic `Field` constraints so that every measurement must be greater than 0 and at most 10 cm. Using the TestClient, show one successful prediction and show that a request with a negative measurement is rejected with status code 422. Finally, run the service with uvicorn and take a screenshot of the `/docs` page showing a successful prediction in the **Response body** panel.

In [22]:

!pip install -q fastapi uvicorn httpx scikit-learn pandas joblib requests

In [23]:
import os
import joblib
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns=[
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width"
    ]
)

y = iris.target

print("Dataset Shape:", X.shape)
print("Flower Species:", iris.target_names)

display(X.head())

Dataset Shape: (150, 4)
Flower Species: ['setosa' 'versicolor' 'virginica']


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Model Accuracy (%):", round(accuracy * 100, 2))

Model Accuracy: 0.9666666666666667
Model Accuracy (%): 96.67


In [25]:
os.makedirs("task", exist_ok=True)

model_path = "task/iris_model.joblib"

joblib.dump(model, model_path)

print("Model saved successfully!")
print("Model Path:", model_path)

Model saved successfully!
Model Path: task/iris_model.joblib


In [26]:
%%writefile task/iris_app.py
import os
import joblib
import pandas as pd

from fastapi import FastAPI
from pydantic import BaseModel, Field

model_path = os.path.join(
    os.path.dirname(__file__),
    "iris_model.joblib"
)

model = joblib.load(model_path)

app = FastAPI(
    title="Iris Flower Classification Service",
    description="Predict Iris flower species using Logistic Regression",
    version="1.0.0"
)

species_names = ["setosa", "versicolor", "virginica"]

class IrisInput(BaseModel):
    sepal_length: float = Field(gt=0, le=10)
    sepal_width: float = Field(gt=0, le=10)
    petal_length: float = Field(gt=0, le=10)
    petal_width: float = Field(gt=0, le=10)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(flower: IrisInput):

    data = pd.DataFrame(
        [flower.model_dump()],
        columns=[
            "sepal_length",
            "sepal_width",
            "petal_length",
            "petal_width"
        ]
    )

    prediction = model.predict(data)[0]

    return {
        "predicted_species": species_names[int(prediction)]
    }

Overwriting task/iris_app.py


In [27]:
import sys
import importlib

from fastapi.testclient import TestClient

if "task" not in sys.path:
    sys.path.insert(0, "task")

import iris_app
importlib.reload(iris_app)

client = TestClient(iris_app.app)

response = client.get("/health")

print("Status Code:", response.status_code)
print("Response:", response.json())

assert response.status_code == 200
assert response.json() == {"status": "ok"}

print("Health endpoint test passed!")

Status Code: 200
Response: {'status': 'ok'}
Health endpoint test passed!


In [28]:
flower_data = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

response = client.post(
    "/predict",
    json=flower_data
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

assert response.status_code == 200
assert response.json()["predicted_species"] == "setosa"

print("Successful prediction test passed!")

Status Code: 200
Prediction: {'predicted_species': 'setosa'}
Successful prediction test passed!


In [29]:
invalid_flower = {
    "sepal_length": -5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

response = client.post(
    "/predict",
    json=invalid_flower
)

print("Status Code:", response.status_code)
print("Error:", response.json()["detail"][0]["msg"])

assert response.status_code == 422

print("Invalid input rejected successfully!")

Status Code: 422
Error: Input should be greater than 0
Invalid input rejected successfully!


In [30]:
import subprocess
import sys
import time
import requests

# Stop an Iris server previously started by this cell
if "iris_server" in globals() and iris_server.poll() is None:
    iris_server.terminate()
    iris_server.wait()

# Start Iris API
iris_server = subprocess.Popen(
    [
        sys.executable,
        "-m", "uvicorn",
        "iris_app:app",
        "--host", "0.0.0.0",
        "--port", "8001"
    ],
    cwd="task",
    stdout=open("/tmp/iris_8001.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

print("Iris Server Running:", iris_server.poll() is None)

if iris_server.poll() is not None:
    print(open("/tmp/iris_8001.log").read())

Iris Server Running: True


In [31]:
import requests

response = requests.get(
    "http://127.0.0.1:8001/openapi.json",
    timeout=10
)

print("Status Code:", response.status_code)

api_title = response.json()["info"]["title"]

print("API Title:", api_title)

assert response.status_code == 200
assert api_title == "Iris Flower Classification Service"

print("Correct Iris API is running!")

Status Code: 200
API Title: Iris Flower Classification Service
Correct Iris API is running!


In [32]:
import requests

flower_data = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

response = requests.post(
    "http://127.0.0.1:8001/predict",
    json=flower_data,
    timeout=10
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

assert response.status_code == 200
assert response.json()["predicted_species"] == "setosa"

print("Live Iris API test passed!")

Status Code: 200
Prediction: {'predicted_species': 'setosa'}
Live Iris API test passed!


In [33]:
from google.colab import output
from IPython.display import display, HTML

url = output.eval_js(
    "google.colab.kernel.proxyPort(8001)"
)

docs_url = url.rstrip("/") + "/docs"

print("Swagger UI URL:", docs_url)

display(HTML(
    f'<a href="{docs_url}" target="_blank">'
    'CLICK HERE TO OPEN IRIS FASTAPI DOCS'
    '</a>'
))

Swagger UI URL: https://8001-m-s-use1d9-2ln6zv0waiiba-d.us-east1.prod.r.colab.dev/docs
